<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/19-efficient-scalable-training.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Efficient and Scalable Training** {#efficient-scalable-training}

Efficient training is the discipline of converting a fixed hardware, time, energy, and reliability budget into trustworthy optimization progress. A method is not efficient merely because one kernel is fast: input stalls, activation memory, gradient communication, recompilation, checkpoint pauses, and failed jobs all contribute to time-to-result. **Scalable** means that adding resources increases feasible model/data size or reduces completion time without unacceptable utilization loss or numerical drift.

The chapter follows one reproducible workload: scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) dataset. It contains 1,797 normalized 8×8 images and ten classes; the UCI release is licensed **CC BY 4.0**. A deep residual MLP is deliberately larger than the task requires so memory, microbatching, graph capture, routing, profiling, and checkpoint state remain visible on CPU. These experiments validate mechanisms, not GPU-cluster speedups.

![One example from each class in the UCI Optical Recognition of Handwritten Digits workload.](assets/dl19-digits-grid.png){fig-align="center" width="66%" fig-alt="A two-row grid shows one 8 by 8 grayscale handwritten image for each digit class from zero through nine."}

*Data source: Alpaydin and Kaynak, [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49), CC BY 4.0. The displayed samples come from scikit-learn's documented `load_digits` copy.*

<details>
<summary><strong>PyTorch: establish the shared Digits workload and deterministic split</strong></summary>

```python
import copy
import io
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1919):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_indices = np.arange(len(digits.data))
train_ids, remaining_ids = train_test_split(
    all_indices, test_size=0.30, stratify=digits.target, random_state=1919
)
val_ids, test_ids = train_test_split(
    remaining_ids, test_size=0.50, stratify=digits.target[remaining_ids], random_state=1919
)

# The fixed 0..16 pixel scale is documented by UCI; no test statistic is fitted.
features = torch.tensor(digits.data / 16.0, dtype=torch.float32)
targets = torch.tensor(digits.target, dtype=torch.long)
train_dataset = TensorDataset(features[train_ids], targets[train_ids])
val_dataset = TensorDataset(features[val_ids], targets[val_ids])
test_dataset = TensorDataset(features[test_ids], targets[test_ids])
loader_generator = torch.Generator().manual_seed(1919)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=loader_generator)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)


class ResidualMLPBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.linear1 = nn.Linear(width, 2 * width)
        self.linear2 = nn.Linear(2 * width, width)

    def forward(self, x):
        residual = x
        x = self.linear2(F.gelu(self.linear1(self.norm(x))))
        return residual + 0.25 * x


class DigitMLP(nn.Module):
    def __init__(self, width=128, depth=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(64, width), nn.GELU())
        self.blocks = nn.Sequential(*[ResidualMLPBlock(width) for _ in range(depth)])
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 10))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))


def accuracy(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            correct += int((model(x).argmax(1) == y).sum())
            total += len(y)
    return correct / total


seed_everything()
baseline_model = DigitMLP()
baseline_optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=2e-3, weight_decay=1e-4)
for _ in range(10):
    baseline_model.train()
    for x, y in train_loader:
        loss = F.cross_entropy(baseline_model(x), y)
        baseline_optimizer.zero_grad(); loss.backward(); baseline_optimizer.step()

batch_inputs, batch_targets = next(iter(DataLoader(train_dataset, batch_size=64, shuffle=False)))
parameter_count = sum(parameter.numel() for parameter in baseline_model.parameters())
assert len(set(train_ids) & set(test_ids)) == 0 and batch_inputs.shape == (64, 64)
print({"split": (len(train_ids), len(val_ids), len(test_ids)), "parameters": parameter_count, "test accuracy": round(accuracy(baseline_model, test_loader), 3)})
```

</details>

The fixed split, batch, model definition, and trained baseline are reused from top to bottom. Hardware-specific APIs are introduced with guarded production patterns; CPU simulations only claim algebraic equivalence or resource accounting.

### **Accelerators and the Training Cost Model** {#accelerators-training-cost-model}

Accelerators are effective because dense linear algebra exposes parallel multiply-accumulate work and reuses data in fast on-chip storage. Their advertised peak FLOP/s is only a ceiling. A training step also reads parameters and activations, writes intermediates, launches kernels, synchronizes devices, and waits for input. The first task is therefore to identify the limiting resource.

For an operation with $F$ useful floating-point operations and $Q$ bytes transferred from the limiting memory level, arithmetic intensity is $I=F/Q$. The roofline bound is

$$
P_{\mathrm{attainable}}\leq\min(P_{\mathrm{peak}},\;I\,B_{\mathrm{memory}}),
$$

where $P_{\mathrm{peak}}$ is peak compute throughput and $B_{\mathrm{memory}}$ memory bandwidth. Distributed execution adds communication time. A simple non-overlapped step model is

$$
T_{\mathrm{step}}\approx T_{\mathrm{input}}+T_{\mathrm{compute}}+T_{\mathrm{communication}}+T_{\mathrm{optimizer}}+T_{\mathrm{checkpoint/amortized}}.
$$

![The roofline model separates memory-bound and compute-bound operating regions.](assets/dl19-roofline.svg){fig-align="center" width="73%" fig-alt="A roofline chart rises with arithmetic intensity in the bandwidth-bound region and becomes flat at the peak compute ceiling."}

Model FLOPs utilization (MFU) divides estimated useful model FLOPs per second by hardware peak FLOP/s. It is useful only when the FLOP convention, precision, sparsity, and peak specification match. End-to-end samples/s or tokens/s remains the operational metric because it includes input and synchronization costs.

<details>
<summary><strong>Python: estimate the workload's linear-layer FLOPs and a roofline bound</strong></summary>

```python
def linear_forward_flops(model, batch_size):
    # A dense [B, in] @ [in, out] multiply is approximately 2*B*in*out FLOPs.
    return sum(
        2 * batch_size * module.in_features * module.out_features
        for module in model.modules() if isinstance(module, nn.Linear)
    )


forward_flops = linear_forward_flops(baseline_model, len(batch_inputs))
training_flops = 3 * forward_flops  # forward + two dominant backward matrix products
parameter_bytes = sum(parameter.numel() * parameter.element_size() for parameter in baseline_model.parameters())
approximate_bytes = parameter_bytes + batch_inputs.numel() * batch_inputs.element_size()
arithmetic_intensity = training_flops / approximate_bytes

# A transparent hypothetical device, not a benchmark of the current CPU.
peak_flops_per_second = 10e12
memory_bytes_per_second = 200e9
roofline = min(peak_flops_per_second, arithmetic_intensity * memory_bytes_per_second)
assert roofline <= peak_flops_per_second
print({"training FLOPs/batch": training_flops, "approx intensity": round(arithmetic_intensity, 1), "hypothetical roofline TFLOP/s": round(roofline / 1e12, 2)})
```

</details>

The byte estimate omits activations, cache reuse, optimizer traffic, and workspaces, so it is a lower-fidelity diagnostic rather than a performance prediction. Real diagnosis requires profiler traces and measured device counters.

### **A Training Memory Ledger** {#training-memory-ledger}

“The model fits” is not a sufficient memory calculation. Peak memory is the maximum simultaneous footprint of parameters, master weights, gradients, optimizer state, saved activations, temporary operator workspaces, communication buckets, and allocator fragmentation.

![A training memory ledger separates static model state, activations, and transient allocations.](assets/dl19-memory-ledger.svg){fig-align="center" width="76%" fig-alt="A stacked ledger contains parameters, gradients, optimizer state, saved activations, and transient workspaces or communication buckets."}

For $P$ trainable parameter elements, a static ledger is

$$
M_{\mathrm{static}}=P(b_w+b_g+b_{m}+b_v+b_{\mathrm{master}}),
$$

where each $b$ is bytes per element and absent copies contribute zero. FP32 Adam commonly uses 4-byte weights, 4-byte gradients, and two 4-byte moments: about $16P$ bytes before activations. Mixed-precision implementations may add FP32 master weights while storing low-precision model weights and gradients, so “half precision halves memory” is generally false.

Activation memory depends on batch size, sequence/image shape, layer widths, attention pattern, and what autograd saves. It can exceed model state for long sequences. `nvidia-smi` reports process-level allocation, while framework counters distinguish live tensors, reserved allocator blocks, and peaks; non-framework CUDA allocations may require separate tools.

<details>
<summary><strong>PyTorch: account for parameter state and tensors saved for backward</strong></summary>

```python
def tensor_bytes(tensor):
    return tensor.numel() * tensor.element_size()


fp32_parameter_bytes = sum(tensor_bytes(parameter) for parameter in baseline_model.parameters())
fp32_adam_static_bytes = 4 * fp32_parameter_bytes  # weights + gradients + two moments
saved_sizes = []


def pack_hook(tensor):
    saved_sizes.append(tensor_bytes(tensor))
    return tensor


def unpack_hook(tensor):
    return tensor


probe_model = copy.deepcopy(baseline_model).train()
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    probe_loss = F.cross_entropy(probe_model(batch_inputs), batch_targets)
    probe_loss.backward()

saved_for_backward_bytes = sum(saved_sizes)
assert fp32_adam_static_bytes > fp32_parameter_bytes and saved_for_backward_bytes > 0
print({"parameter MiB": round(fp32_parameter_bytes / 2**20, 3), "FP32 Adam static MiB": round(fp32_adam_static_bytes / 2**20, 3), "saved tensors MiB": round(saved_for_backward_bytes / 2**20, 3), "saved tensor count": len(saved_sizes)})
```

</details>

Saved-tensor hooks count logical tensors requested by autograd, not allocator peak memory; aliasing and temporary kernels complicate the physical footprint. The method is useful for explaining scaling, while CUDA memory snapshots and profiler traces are required for exact diagnosis.

### **FP32, FP16, BF16, and FP8** {#floating-point-formats}

A floating-point format allocates bits to sign, exponent, and fraction. Exponent bits determine dynamic range; fraction bits determine local precision. FP16 has more fraction bits than BF16 but a much narrower exponent range. BF16 shares FP32's eight exponent bits, making it less prone to overflow and often easier for training, at the cost of coarser rounding. FP8 is not one format: [FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433) defines E4M3 and E5M2 trade-offs, and practical recipes require scaling and higher-precision accumulation.

![FP32, FP16, BF16, and FP8 allocate different numbers of exponent and fraction bits.](assets/dl19-precision-formats.svg){fig-align="center" width="76%" fig-alt="Rows compare sign, exponent, and fraction fields for FP32, FP16, BF16, and FP8 E4M3 or E5M2 formats."}

| Format | Typical role in training | Strength | Main risk |
|---|---|---|---|
| FP32 | master state, reductions, numerically sensitive operations | range and precision | memory and lower tensor-core throughput |
| FP16 | matrix operations and activations | compact with good precision near one | overflow and gradient underflow |
| BF16 | matrix operations and activations | FP32-like range | fewer fraction bits |
| FP8 E4M3 | forward activations/weights on supported systems | greater density, more precision than E5M2 | limited range and scaling sensitivity |
| FP8 E5M2 | gradients or values needing wider range | wider FP8 range | very coarse precision |

<details>
<summary><strong>PyTorch: inspect range and rounding error across available formats</strong></summary>

```python
formats = [torch.float32, torch.float16, torch.bfloat16]
if hasattr(torch, "float8_e4m3fn"):
    formats += [torch.float8_e4m3fn, torch.float8_e5m2]

values = torch.tensor([1e-4, 0.1, 1.0, 17.25, 123.0], dtype=torch.float32)
precision_report = {}
for dtype in formats:
    info = torch.finfo(dtype)
    restored = values.to(dtype).float()
    precision_report[str(dtype).replace("torch.", "")] = {
        "tiny": float(info.tiny),
        "max": float(info.max),
        "max abs error": float((values - restored).abs().max()),
    }

assert precision_report["float32"]["max abs error"] <= precision_report["bfloat16"]["max abs error"]
print(precision_report)
```

</details>

Casting a tensor is not a complete low-precision training recipe. Matrix inputs, accumulation, normalization, softmax, optimizer state, gradient reduction, and communication may use different formats. Hardware support determines whether lower precision accelerates computation or merely adds conversion overhead.

### **Automatic Mixed Precision and Loss Scaling** {#automatic-mixed-precision-loss-scaling}

Automatic mixed precision (AMP) chooses lower precision for operations that benefit and retains safer precision for sensitive reductions or unsupported kernels. In PyTorch, `autocast` controls operator dtypes; it does not permanently convert the model. The [official AMP examples](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) pair autocast with `GradScaler` for FP16 training.

![AMP combines autocast, loss scaling, unscaling, overflow checks, and the optimizer step.](assets/dl19-amp-flow.svg){fig-align="center" width="76%" fig-alt="FP32 state enters an autocast forward, the loss is scaled for backward, then gradients are unscaled, checked, clipped, and applied."}

If $g$ is a small gradient and $S$ a scale factor, backpropagating $S\mathcal L$ produces $Sg$, moving values away from FP16 underflow. Before clipping or stepping, gradients are divided by $S$. Dynamic scaling increases $S$ after stable steps and decreases it after non-finite gradients. The optimizer step is skipped on overflow, so scheduler updates must follow actual optimizer steps rather than every attempted batch.

BF16 normally does not require loss scaling because of its wider exponent range, although overflow and unstable operations remain possible. Norms, exponentials, reductions, and some losses often accumulate in FP32.

<details>
<summary><strong>PyTorch: run one guarded AMP step on the shared workload</strong></summary>

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_model = copy.deepcopy(baseline_model).to(device).train()
amp_optimizer = torch.optim.AdamW(amp_model.parameters(), lr=1e-3)
amp_inputs, amp_targets = batch_inputs.to(device), batch_targets.to(device)
autocast_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

amp_optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type=device.type, dtype=autocast_dtype):
    amp_logits = amp_model(amp_inputs)
    amp_loss = F.cross_entropy(amp_logits, amp_targets)
scaler.scale(amp_loss).backward()
scaler.unscale_(amp_optimizer)  # clipping must see true-scale gradients
gradient_norm = nn.utils.clip_grad_norm_(amp_model.parameters(), max_norm=1.0)
scaler.step(amp_optimizer)
scaler.update()

assert torch.isfinite(amp_loss) and torch.isfinite(gradient_norm)
print({"device": device.type, "autocast dtype": str(autocast_dtype), "logits dtype": str(amp_logits.dtype), "loss": round(float(amp_loss.detach()), 4), "scaler enabled": scaler.is_enabled()})
```

</details>

CPU BF16 in this example validates the autocast control flow, not accelerator speed. On CUDA, inspect skipped steps, scale history, non-finite counts, loss parity, and task quality before claiming a successful precision migration.

### **Gradient Accumulation, Checkpointing, and Offloading** {#accumulation-checkpointing-offloading}

These techniques relieve different constraints. Gradient accumulation splits a global batch into microbatches and delays the optimizer step. Activation checkpointing discards selected forward intermediates and recomputes them during backward; the [original sublinear-memory work](https://arxiv.org/abs/1604.06174) formalizes this compute-memory trade-off. Offloading moves state or activations to CPU memory or storage and must hide transfer behind useful computation to avoid becoming bandwidth-bound.

![Gradient accumulation, activation checkpointing, and offloading trade different resources for memory.](assets/dl19-memory-techniques.svg){fig-align="center" width="76%" fig-alt="Three panels show accumulated microbatch gradients, recomputed checkpoint regions, and state transferred to CPU or storage."}

With $K$ equal-size microbatches and a mean-reduced loss, each microbatch loss must be divided by $K$ so

$$
\nabla_\theta \mathcal L_{\mathrm{global}}=
\frac{1}{K}\sum_{k=1}^{K}\nabla_\theta\mathcal L_k.
$$

Variable token counts require normalization by total valid tokens rather than number of microbatches. BatchNorm, dropout masks, gradient clipping, optimizer schedules, and DDP synchronization can prevent exact equivalence. In DDP, `no_sync()` suppresses all-reduce on intermediate microbatches.

<details>
<summary><strong>PyTorch: verify gradient accumulation and measure checkpoint recomputation</strong></summary>

```python
from torch.utils.checkpoint import checkpoint_sequential

full_batch_model = copy.deepcopy(baseline_model).train()
accumulated_model = copy.deepcopy(baseline_model).train()
full_loss = F.cross_entropy(full_batch_model(batch_inputs), batch_targets)
full_loss.backward()

microbatch_count = 4
for x_micro, y_micro in zip(batch_inputs.chunk(microbatch_count), batch_targets.chunk(microbatch_count)):
    (F.cross_entropy(accumulated_model(x_micro), y_micro) / microbatch_count).backward()
maximum_gradient_difference = max(
    float((left.grad - right.grad).abs().max())
    for left, right in zip(full_batch_model.parameters(), accumulated_model.parameters())
)

checkpoint_model = copy.deepcopy(baseline_model).train()
block_forward_counts = [0 for _ in checkpoint_model.blocks]
hook_handles = []
for block_index, block in enumerate(checkpoint_model.blocks):
    def count_forward(_module, _inputs, _output, index=block_index):
        block_forward_counts[index] += 1
    hook_handles.append(block.register_forward_hook(count_forward))

def checkpointed_forward(x):
    hidden = checkpoint_model.stem(x)
    hidden = checkpoint_sequential(checkpoint_model.blocks, segments=3, input=hidden, use_reentrant=False)
    return checkpoint_model.head(hidden)

checkpoint_loss = F.cross_entropy(checkpointed_forward(batch_inputs), batch_targets)
checkpoint_loss.backward()
for handle in hook_handles:
    handle.remove()
recomputed_blocks = sum(count > 1 for count in block_forward_counts)
assert maximum_gradient_difference < 1e-5 and torch.isfinite(checkpoint_loss) and recomputed_blocks > 0
print({"accumulation max gradient difference": maximum_gradient_difference, "checkpoint loss": round(float(checkpoint_loss.detach()), 4), "block forward counts": block_forward_counts, "recomputed blocks": recomputed_blocks})
```

</details>

Checkpointed functions must be deterministic under recomputation; PyTorch preserves relevant RNG state by default, with overhead. Moving tensors to a previously unseen device inside the checkpointed function can break equivalence. Offloading introduces a separate correctness problem: stale or late-prefetched state can silently serialize the step.

### **Compilation and Fused Kernels** {#compilation-fused-kernels}

Eager execution launches operators individually and materializes many intermediate tensors. Compilation captures graph regions, specializes them under shape and type guards, and can fuse compatible operations into fewer kernels. Fusion reduces launch overhead and round trips to device memory; it does not reduce the mathematical FLOPs of the model.

![Compilation fuses compatible operators, while graph breaks and guard changes create new regions or recompilations.](assets/dl19-compile-fusion.svg){fig-align="center" width="75%" fig-alt="Bias, GELU, and dropout kernels become one fused region; a graph break divides regions and changing guards triggers recompilation."}

`torch.compile` preserves Python semantics through guards and may compile multiple variants. Dynamic control flow, unsupported operations, Python side effects, changing shapes, and scalar extraction can cause graph breaks or recompilation. First-iteration compile cost must be amortized; short jobs can become slower. The [PyTorch profiler guidance](https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/torch.compiler_profiling_torch_compile.html) recommends inspecting compiled regions and graph breaks rather than assuming compilation succeeded because code ran.

Fused optimizers, fused normalization, and fused attention are library kernels designed for common patterns. They should be preferred over handwritten CUDA unless the profile proves a missing kernel dominates and the maintenance cost is justified.

<details>
<summary><strong>PyTorch: validate graph capture without pretending CPU eager backend is a speed benchmark</strong></summary>

```python
compile_probe = copy.deepcopy(baseline_model).eval()
eager_output = compile_probe(batch_inputs)
compiled_probe = torch.compile(compile_probe, backend="eager", fullgraph=True)
compiled_output = compiled_probe(batch_inputs)

# FX exposes operator structure; backend='eager' checks capture but intentionally performs no fusion.
fx_graph = torch.fx.symbolic_trace(copy.deepcopy(baseline_model).eval())
call_nodes = [node for node in fx_graph.graph.nodes if node.op in {"call_module", "call_function", "call_method"}]
assert torch.allclose(eager_output, compiled_output, atol=1e-6)
print({"captured output parity": True, "FX call nodes": len(call_nodes), "backend": "eager (capture validation only)"})
```

</details>

A production benchmark uses a real optimizing backend, warm-up iterations, synchronization around timing, representative shapes, and quality checks. A lower kernel count can still lose if fusion increases register pressure, reduces occupancy, or repeatedly recompiles.

### **Data Parallelism and DistributedDataParallel** {#data-parallelism-ddp}

Data parallelism places a full model replica on every rank, partitions the global batch, computes local gradients, and averages those gradients. If rank $r$ processes $B_r$ equal-sized examples and the loss is a mean, synchronous DDP computes

$$
g=\frac{1}{W}\sum_{r=1}^{W}g_r,
$$

where $W$ is world size. Unequal local batch or token counts require weighted reduction. PyTorch [DistributedDataParallel](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html) synchronizes gradients but does **not** shard input automatically; a sampler or data service must assign rank-local examples.

![DDP replicas process different batch shards and average bucketed gradients.](assets/dl19-ddp-allreduce.svg){fig-align="center" width="76%" fig-alt="Three ranks process separate data shards, send gradients to a bucketed all-reduce, and receive the averaged gradients."}

DDP registers autograd hooks and groups gradients into buckets so all-reduce can overlap with later backward computation. Bucket size trades launch latency against overlap. `find_unused_parameters=True`, divergent control flow, stragglers, and tiny layers can reduce efficiency. DDP improves throughput only when computation per rank is large enough to amortize synchronization.

<details>
<summary><strong>PyTorch: emulate two DDP ranks and verify averaged-gradient equivalence</strong></summary>

```python
reference_model = copy.deepcopy(baseline_model).train()
reference_loss = F.cross_entropy(reference_model(batch_inputs), batch_targets)
reference_loss.backward()

rank_models = [copy.deepcopy(baseline_model).train() for _ in range(2)]
for rank_model, x_rank, y_rank in zip(rank_models, batch_inputs.chunk(2), batch_targets.chunk(2)):
    F.cross_entropy(rank_model(x_rank), y_rank).backward()

maximum_allreduce_difference = 0.0
for parameter_index, reference_parameter in enumerate(reference_model.parameters()):
    averaged = torch.stack([
        list(rank_model.parameters())[parameter_index].grad for rank_model in rank_models
    ]).mean(0)
    maximum_allreduce_difference = max(
        maximum_allreduce_difference, float((averaged - reference_parameter.grad).abs().max())
    )

assert maximum_allreduce_difference < 1e-5
print({"world size": 2, "max gradient difference": maximum_allreduce_difference, "global batch": len(batch_inputs)})
```

</details>

The emulation validates gradient algebra for equal shards, not collective latency or overlap. A real run also verifies identical initialization, deterministic control flow, rank-local devices, sampler epochs, and that only rank zero performs non-sharded side effects such as ordinary checkpoint writing.

### **FSDP and ZeRO** {#fsdp-zero}

DDP replicates parameters, gradients, and optimizer states, so per-rank model-state memory does not fall as more devices are added. [ZeRO](https://arxiv.org/abs/1910.02054) partitions redundant state across the data-parallel group. ZeRO Stage 1 shards optimizer state, Stage 2 also shards gradients, and Stage 3 also shards parameters. PyTorch [FSDP `FULL_SHARD`](https://docs.pytorch.org/docs/stable/fsdp.html) similarly all-gathers parameter shards before computation and reduce-scatters gradients afterward.

![ZeRO stages progressively shard optimizer state, gradients, and parameters.](assets/dl19-zero-stages.svg){fig-align="center" width="76%" fig-alt="A table shows DDP replicating all model states, ZeRO stage one sharding optimizer state, stage two also gradients, and stage three or FSDP sharding all three."}

With mixed-precision Adam using 2 bytes for model weights, 2 for gradients, 4 for a master copy, and 8 for moments, a simplified per-rank ledger is

$$
M_{\mathrm{DDP}}\approx16P,\quad
M_{Z1}\approx4P+\frac{12P}{W},\quad
M_{Z2}\approx2P+\frac{14P}{W},\quad
M_{Z3}\approx\frac{16P}{W}.
$$

Activations, buffers, temporary all-gathered parameters, communication buckets, fragmentation, and checkpoint staging are excluded. Sharding lowers residency but increases collectives and makes wrapping granularity important: tiny FSDP units create many collectives; huge units create large memory spikes and weak overlap.

<details>
<summary><strong>Python: calculate sharded state memory and reconstruct a parameter shard</strong></summary>

```python
def model_state_bytes_per_rank(parameters, world_size):
    return {
        "DDP": 16 * parameters,
        "ZeRO-1": 4 * parameters + 12 * parameters / world_size,
        "ZeRO-2": 2 * parameters + 14 * parameters / world_size,
        "ZeRO-3/FSDP": 16 * parameters / world_size,
    }


world_size = 4
actual_ledger = model_state_bytes_per_rank(parameter_count, world_size)
seven_billion_ledger_gib = {
    name: value / 2**30 for name, value in model_state_bytes_per_rank(7_000_000_000, world_size).items()
}

flat_weight = baseline_model.stem[0].weight.detach().flatten()
shards = list(torch.tensor_split(flat_weight, world_size))
reconstructed = torch.cat(shards)
assert torch.equal(flat_weight, reconstructed) and actual_ledger["ZeRO-3/FSDP"] < actual_ledger["DDP"]
print({"actual model MiB/rank": {k: round(v / 2**20, 3) for k, v in actual_ledger.items()}, "7B model GiB/rank (state only)": {k: round(v, 2) for k, v in seven_billion_ledger_gib.items()}})
```

</details>

Full-state checkpoints can temporarily gather an entire model and cause rank-zero OOM. Sharded state dictionaries and distributed checkpointing avoid that peak and support resharding across a different world size.

### **Tensor and Sequence Parallelism** {#tensor-sequence-parallelism}

Tensor parallelism partitions the computation of a single layer. For $Y=XW^\top+b$, column parallelism splits rows of $W$ (output features), computes independent output slices, and concatenates them. Row parallelism splits input features and columns of $W$, computes partial outputs, and sums them with an all-reduce.

![Column tensor parallelism partitions output features and concatenates local matrix products.](assets/dl19-tensor-parallel.svg){fig-align="center" width="74%" fig-alt="An input matrix is multiplied by two output-column weight shards, and local outputs are concatenated into the complete result."}

[Megatron-LM](https://arxiv.org/abs/1909.08053) arranges Transformer attention and MLP projections so collectives occur at controlled boundaries. Tensor parallelism reduces per-rank parameter and activation residency within a layer but introduces latency-sensitive communication every layer. It therefore favors high-bandwidth links within a node.

Sequence parallelism shards activations along the sequence dimension for operations that need not retain a full sequence on each tensor-parallel rank, such as some normalization and dropout regions. It reduces replicated activation memory; it is distinct from context parallelism, which partitions attention context and must exchange key/value information or partial attention statistics.

<details>
<summary><strong>PyTorch: reproduce one trained linear layer with column and row partitions</strong></summary>

```python
linear = baseline_model.stem[0]
x = batch_inputs
full_output = F.linear(x, linear.weight, linear.bias)

# Column parallel: split output rows, compute locally, concatenate features.
weight_output_shards = linear.weight.chunk(2, dim=0)
bias_shards = linear.bias.chunk(2, dim=0)
column_output = torch.cat([
    F.linear(x, weight_shard, bias_shard)
    for weight_shard, bias_shard in zip(weight_output_shards, bias_shards)
], dim=-1)

# Row parallel: split input features and weight columns, sum partial outputs, add bias once.
input_shards = x.chunk(2, dim=-1)
weight_input_shards = linear.weight.chunk(2, dim=1)
row_output = sum(F.linear(input_shard, weight_shard, None) for input_shard, weight_shard in zip(input_shards, weight_input_shards)) + linear.bias

assert torch.allclose(full_output, column_output, atol=1e-6)
assert torch.allclose(full_output, row_output, atol=1e-6)
print({"full shape": tuple(full_output.shape), "column local shape": tuple(column_output[:, : column_output.shape[1] // 2].shape), "row collective": "sum/all-reduce"})
```

</details>

These local tensor operations prove partition equivalence. They do not model collective order, asynchronous overlap, topology, or numerical differences from reduction order.

### **Pipeline and Context Parallelism** {#pipeline-context-parallelism}

Pipeline parallelism assigns consecutive model stages to different devices and splits a batch into $m$ microbatches. While one stage handles a later microbatch, another handles an earlier one. In a simple all-forward then all-backward GPipe schedule with $p$ stages, the idealized forward utilization is

$$
\eta_{\mathrm{pipeline}}\approx\frac{m}{m+p-1}.
$$

![A pipeline fills, overlaps microbatches across stages, and drains.](assets/dl19-pipeline-context.svg){fig-align="center" width="75%" fig-alt="Three model stages process four microbatches on an offset timeline with fill and drain bubbles."}

Increasing $m$ shrinks the bubble fraction but makes microbatches smaller and increases scheduling overhead. Stage partitioning must balance compute time and activation transfer, not layer count. Interleaved and 1F1B schedules reduce idle time or activation residency but make dependencies more complex. [GPipe](https://arxiv.org/abs/1811.06965) is the canonical batch-splitting pipeline reference.

Context parallelism addresses sequences too long for one device's attention activations. It partitions tokens across ranks and exchanges key/value blocks or partial softmax statistics. Correct distributed softmax requires globally consistent maxima and normalization sums; naively attending only to local tokens changes the model.

<details>
<summary><strong>PyTorch: partition the Digits MLP into stages and calculate pipeline bubbles</strong></summary>

```python
baseline_model.eval()
block_list = list(baseline_model.blocks.children())
stage_0 = nn.Sequential(baseline_model.stem, *block_list[:3])
stage_1 = nn.Sequential(*block_list[3:], baseline_model.head)
staged_output = stage_1(stage_0(batch_inputs))
direct_output = baseline_model(batch_inputs)

pipeline_efficiency = {
    microbatches: microbatches / (microbatches + 3 - 1)
    for microbatches in (1, 2, 4, 8, 16)
}
assert torch.allclose(staged_output, direct_output, atol=1e-6)
print({"stage boundary": tuple(stage_0(batch_inputs).shape), "three-stage idealized efficiency": {k: round(v, 3) for k, v in pipeline_efficiency.items()}})
```

</details>

The code verifies layer partitioning but executes serially. Real pipeline correctness also requires consistent microbatch loss normalization, tied-weight handling, deterministic recomputation, and matching optimizer semantics across stages.

### **Expert Parallelism and Mixture-of-Experts** {#expert-parallelism-moe}

A mixture-of-experts (MoE) layer contains many parameterized experts but routes each token to only the top $k$. This increases parameter capacity without applying every parameter to every token. Expert parallelism places different experts on different ranks; token states are exchanged with all-to-all collectives, processed, and returned to their original sequence positions.

![An MoE router dispatches tokens to capacity-limited experts and combines their outputs.](assets/dl19-moe-routing.svg){fig-align="center" width="74%" fig-alt="Token states enter a top-k router, travel to experts on different device groups, and return to a weighted combine stage."}

Let $p_{te}$ be the router probability that token $t$ selects expert $e$. Top-1 routing chooses $e_t=\arg\max_e p_{te}$. A capacity factor $c$ often limits each expert to roughly $\lceil cT/E\rceil$ tokens. Skew creates hot experts, dropped or rerouted tokens, and rank stragglers. Auxiliary load-balancing losses encourage both routing probability and actual assignments to spread across experts.

The [Switch Transformer](https://arxiv.org/abs/2101.03961) simplifies routing to one expert per token, but sparse compute does not mean cheap communication. Expert placement, token permutation, capacity, topology, and numerical stability determine realized throughput.

<details>
<summary><strong>PyTorch: route shared-workload representations through capacity-limited experts</strong></summary>

```python
seed_everything(1926)
with torch.no_grad():
    token_states = baseline_model.stem(batch_inputs)
expert_count = 4
router = nn.Linear(token_states.shape[-1], expert_count, bias=False)
experts = nn.ModuleList([
    nn.Sequential(nn.Linear(128, 192), nn.GELU(), nn.Linear(192, 128))
    for _ in range(expert_count)
])
routing_probabilities = router(token_states).softmax(-1)
assignments = routing_probabilities.argmax(-1)
capacity = math.ceil(1.25 * len(token_states) / expert_count)
moe_output = token_states.clone()
expert_loads, overflow = [], 0
for expert_id, expert in enumerate(experts):
    token_ids = (assignments == expert_id).nonzero(as_tuple=False).squeeze(1)
    expert_loads.append(len(token_ids))
    accepted, rejected = token_ids[:capacity], token_ids[capacity:]
    if len(accepted):
        moe_output[accepted] = expert(token_states[accepted])
    overflow += len(rejected)  # fallback keeps the residual representation in this demonstration

load_fraction = torch.bincount(assignments, minlength=expert_count).float() / len(assignments)
assert moe_output.shape == token_states.shape and sum(expert_loads) == len(token_states)
print({"expert loads": expert_loads, "capacity": capacity, "overflow": overflow, "load coefficient of variation": round(float(load_fraction.std() / load_fraction.mean()), 3)})
```

</details>

This untrained router intentionally exposes imbalance. A production MoE trains routing jointly, includes weighted top-$k$ combination and auxiliary losses, and measures per-expert token counts plus all-to-all time rather than reporting parameter count alone.

### **Distributed Data Pipelines** {#distributed-data-pipelines}

An accelerator cannot compensate for a pipeline that decodes, augments, or transfers data too slowly. The path includes storage layout, sharding, shuffle, CPU workers, transforms, collation, pinned memory, host-to-device transfer, and device prefetch. Each rank must receive a distinct, statistically valid sample stream and reach collective calls at compatible times.

![A distributed input pipeline overlaps storage reads, CPU transforms, pinned buffers, transfer, and device compute.](assets/dl19-data-pipeline.svg){fig-align="center" width="75%" fig-alt="Storage feeds CPU workers, a pinned batch buffer, asynchronous device transfer, and compute, with preparation overlapped across batches."}

PyTorch's [data-loading documentation](https://docs.pytorch.org/docs/stable/data.html) distinguishes map-style and iterable datasets. `DistributedSampler` partitions indices but must receive `set_epoch(epoch)` for a new deterministic shuffle. If dataset size is not divisible by world size, padding can duplicate samples; `drop_last=True` instead omits a tail. Iterable datasets require explicit rank and worker sharding to avoid every process reading the same stream.

Pinned host memory permits asynchronous DMA to CUDA when combined with `non_blocking=True`, but pinning too much memory harms the host. Worker count, prefetch depth, and persistent workers should be tuned from measured queue stalls, not maximized blindly.

<details>
<summary><strong>PyTorch: inspect rank-local sampling and epoch-dependent shuffling</strong></summary>

```python
from torch.utils.data.distributed import DistributedSampler

sampler_indices = {}
for rank in range(4):
    sampler = DistributedSampler(
        train_dataset, num_replicas=4, rank=rank, shuffle=True, seed=1919, drop_last=True
    )
    sampler.set_epoch(0)
    sampler_indices[rank] = list(iter(sampler))

rank_sets = [set(indices) for indices in sampler_indices.values()]
pairwise_overlap = max(len(rank_sets[left] & rank_sets[right]) for left in range(4) for right in range(left + 1, 4))
covered = len(set().union(*rank_sets))

epoch_probe = DistributedSampler(train_dataset, num_replicas=4, rank=0, shuffle=True, seed=1919, drop_last=True)
epoch_probe.set_epoch(0); epoch_zero = list(iter(epoch_probe))
epoch_probe.set_epoch(1); epoch_one = list(iter(epoch_probe))
assert pairwise_overlap == 0 and epoch_zero != epoch_one
print({"examples/rank": len(epoch_zero), "covered": covered, "dropped tail": len(train_dataset) - covered, "rank overlap": pairwise_overlap})
```

</details>

For sequence corpora, equal example counts can still produce severe token-count imbalance. Length-aware packing improves utilization but may change sample order and gradient statistics; the exact packing and resume cursor belong in the experiment record.

### **Profiling, Communication, and Bottleneck Diagnosis** {#profiling-communication-bottlenecks}

Optimization should begin with a timeline of representative steady-state steps. A profile distinguishes input gaps, host launch overhead, device kernels, memory allocation, communication, optimizer work, graph breaks, and checkpoint pauses. End-to-end throughput tells **that** a run is slow; a trace helps explain **why**.

![A step timeline separates CPU launch, accelerator kernels, network collectives, and idle gaps.](assets/dl19-profiler-timeline.svg){fig-align="center" width="76%" fig-alt="CPU, GPU, and network tracks show input launch, forward, backward, gradient all-reduce, optimizer work, and idle gaps."}

For a collective message of $n$ bytes, the alpha-beta model writes $T\approx\alpha N_{\mathrm{messages}}+\beta n$, where $\alpha$ captures latency and $\beta$ inverse bandwidth. Small buckets pay latency repeatedly; huge buckets delay overlap. The slowest rank determines synchronous step time, so per-rank traces and straggler percentiles matter more than an average trace.

The [PyTorch Profiler](https://docs.pytorch.org/docs/stable/profiler.html) records operator time, shapes, memory, stacks, and device activities. Warm-up and active windows avoid initialization noise; distributed traces require aligned clocks and rank labels.

<details>
<summary><strong>PyTorch: profile one complete CPU training step on the shared model</strong></summary>

```python
from torch.profiler import ProfilerActivity, profile

profiled_model = copy.deepcopy(baseline_model).train()
profiled_optimizer = torch.optim.SGD(profiled_model.parameters(), lr=1e-3)
with profile(activities=[ProfilerActivity.CPU], record_shapes=True, profile_memory=True, acc_events=True) as trace:
    with torch.profiler.record_function("digit_training_step"):
        profiled_loss = F.cross_entropy(profiled_model(batch_inputs), batch_targets)
        profiled_optimizer.zero_grad(set_to_none=True)
        profiled_loss.backward()
        profiled_optimizer.step()

top_events = trace.key_averages().table(sort_by="self_cpu_time_total", row_limit=5)
assert "Self CPU" in top_events and torch.isfinite(profiled_loss)
print(top_events)
```

</details>

A one-step CPU trace demonstrates instrumentation, not stable ranking. Production profiling discards compilation and allocator warm-up, samples several iterations, synchronizes around manual timers, and correlates framework events with device and network counters.

### **Checkpointing, Fault Tolerance, and Reproducibility** {#checkpointing-fault-tolerance-reproducibility}

A resumable checkpoint contains more than model weights. Exact continuation may require optimizer moments, scheduler, AMP scaler, global step, epoch and data cursor, sampler state, RNG state for every rank, model configuration, tokenizer/data version, and software commit. Saving only weights creates a warm restart with different optimization dynamics.

![A reliable checkpoint combines model, optimizer, schedule, RNG, and data-cursor state under a versioned manifest.](assets/dl19-checkpoint-state.svg){fig-align="center" width="76%" fig-alt="Model, optimizer, scheduler and scaler, random states, and data cursor feed a versioned atomic checkpoint manifest."}

Checkpoint interval balances write overhead against expected lost work. If failures are independent with mean time between failures $M$ and one checkpoint costs $C$, a classical approximation chooses an interval on the order of $\sqrt{2CM}$, then adjusts for restart cost and correlated failures. Asynchronous saving reduces pauses only if staging memory and storage bandwidth do not interfere with training.

Sharded models should avoid gathering a full checkpoint on one rank. PyTorch [Distributed Checkpoint](https://docs.pytorch.org/tutorials/recipes/distributed_checkpoint_recipe.html) writes rank-local shards in parallel and can reshard when loading under another topology. Atomic manifests, checksums, retention policy, and periodic restore tests are part of correctness.

<details>
<summary><strong>PyTorch: serialize and restore a complete in-memory training state</strong></summary>

```python
checkpoint_model = copy.deepcopy(baseline_model).train()
checkpoint_optimizer = torch.optim.AdamW(checkpoint_model.parameters(), lr=1e-3)
checkpoint_loss = F.cross_entropy(checkpoint_model(batch_inputs), batch_targets)
checkpoint_optimizer.zero_grad(); checkpoint_loss.backward(); checkpoint_optimizer.step()
checkpoint_model.eval()
with torch.no_grad():
    checkpoint_logits = checkpoint_model(batch_inputs).clone()

state = {
    "model": checkpoint_model.state_dict(),
    "optimizer": checkpoint_optimizer.state_dict(),
    "step": 1,
    "torch_rng": torch.get_rng_state(),
    "numpy_rng": np.random.get_state(),
    "python_rng": random.getstate(),
    "loader_generator": loader_generator.get_state(),
    "split_ids": {"train": train_ids, "validation": val_ids, "test": test_ids},
}
buffer = io.BytesIO()
torch.save(state, buffer)
buffer.seek(0)
restored_state = torch.load(buffer, weights_only=False)

restored_model = DigitMLP()
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=1e-3)
restored_model.load_state_dict(restored_state["model"])
restored_optimizer.load_state_dict(restored_state["optimizer"])
torch.set_rng_state(restored_state["torch_rng"])
np.random.set_state(restored_state["numpy_rng"])
random.setstate(restored_state["python_rng"])
loader_generator.set_state(restored_state["loader_generator"])
restored_model.eval()

with torch.no_grad():
    restored_logits = restored_model(batch_inputs)
assert torch.equal(checkpoint_logits, restored_logits) and restored_state["step"] == 1
print({"checkpoint KiB": round(buffer.getbuffer().nbytes / 2**10, 1), "optimizer state entries": len(restored_state["optimizer"]["state"]), "exact output restore": True})
```

</details>

Reproducibility has levels: bitwise replay, statistically equivalent training, and comparable final quality are different promises. Collective reduction order, kernel selection, asynchronous execution, worker scheduling, and hardware can prevent bitwise identity even with all seeds saved. Report the level actually tested.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

| Technique | Primary bottleneck addressed | Main trade-off | Validation signal |
|---|---|---|---|
| Mixed precision | tensor compute and tensor storage | numerical range/rounding | loss parity, overflow rate, task quality |
| Gradient accumulation | batch activation memory | more microsteps and delayed update | gradient equivalence and samples/s |
| Activation checkpointing | saved activations | recomputation | peak memory and step time |
| Offloading | device memory capacity | transfer bandwidth and latency | overlap and stall time |
| Compilation/fusion | launch overhead and memory traffic | compile cost and graph specialization | steady-state trace and recompiles |
| DDP | data throughput | replicated model state and all-reduce | scaling efficiency and stragglers |
| FSDP/ZeRO | replicated model state | all-gather/reduce-scatter traffic | peak memory plus communication overlap |
| Tensor parallelism | layer parameter/activation capacity | per-layer collectives | topology-aware throughput |
| Pipeline parallelism | model depth capacity | bubbles and stage imbalance | stage utilization and activation residency |
| Context parallelism | long-sequence activation capacity | distributed attention communication | exactness and sequence throughput |
| Expert parallelism | sparse model capacity | routing imbalance and all-to-all | per-expert load and token throughput |

A practical optimization order is:

1. Freeze a correct single-device baseline, dataset split, metric, and numerical tolerance.
2. Measure end-to-end throughput, peak memory, and a representative steady-state trace.
3. Use AMP and efficient kernels when hardware supports them; verify quality and overflow behavior.
4. Apply accumulation or checkpointing only to the memory category that actually dominates.
5. Scale with DDP while the full model fits and all-reduce can overlap.
6. Introduce FSDP/ZeRO when replicated state is the capacity limit.
7. Add tensor, pipeline, context, or expert parallelism according to the dimension that no longer fits.
8. Rebalance data, compute, and communication after every topology change.
9. Save complete, versioned state and run restore drills before long jobs.
10. Report time-to-quality, utilization, memory, energy/cost assumptions, failure recovery, and numerical parity together.

Scalability is a systems property, not an API flag. Precision, memory lifetime, graph structure, collective topology, data ordering, and failure recovery interact; improving one isolated metric can make time-to-result worse. The next chapter follows the trained model into efficient inference and deployment, where latency, batching, caching, quantization, and service-level objectives replace backward-pass constraints.